# Base, Instruct, and Reasoning: inspect the behavior

This companion investigates two questions: how related base and post-trained checkpoints respond to requests, and what changes when one post-trained checkpoint switches thinking modes. Read the explanations and saved results without running a model, or restart the kernel and execute all cells to reproduce the experiment.

We use **two checkpoints and three conditions**, not three mutually exclusive model types. This is an illustrative experiment, not a benchmark or an isolated causal test of instruction tuning. The article is planned.

## 1. Questions, tasks, and expectations

A **checkpoint** is a saved set of model parameters. **Pretraining** learns from a broad text mixture. **Post-training** adapts the checkpoint through subsequent training. Qwen3's post-trained model supports both instruction following and extended thinking.

The base checkpoint already includes math, code, and reasoning data in its pretraining mixture. Do not interpret it as a model with no reasoning exposure. The post-trained checkpoint differs through multiple stages; output differences cannot be attributed to SFT alone.

We define four tasks and their answers before generation. Every task requests JSON so correctness and output compliance can be checked separately. These are deliberately small synthetic examples, not a representative test set. No generated code is executed.

In [1]:
import gc, json, os, platform, re, time
from datetime import datetime, timezone
from pathlib import Path
from importlib.metadata import version
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from IPython.display import display, Markdown

TASKS = [
 {"id":"extract_order", "kind":"extraction", "prompt":'Extract the order ID and revised delivery time from: "Order Z17 was due at 14:00; it will now arrive at 14:30." Return only JSON with keys order_id and revised_time.', "expected":{"order_id":"Z17","revised_time":"14:30"}},
 {"id":"extract_missing", "kind":"extraction", "prompt":'Extract the order ID and revised delivery time from: "Order K82 is delayed. A new arrival time has not been given." Use null for missing information. Return only JSON with keys order_id and revised_time.', "expected":{"order_id":"K82","revised_time":None}},
 {"id":"box_count", "kind":"arithmetic", "prompt":'A van starts with 12 boxes. The driver unloads 5 boxes at a delivery, then picks up 3 additional boxes and loads them onto the van. How many boxes are on the van now? Return only JSON with key boxes and an integer value.', "expected":{"boxes":10}},
 {"id":"visit_order", "kind":"constraints", "prompt":'Visit A, B, C, and D exactly once. A must be before B. C must be before A. D must be after B. Return only JSON with key route and a list of the four letters in a valid order.', "expected":{"route":["C","A","B","D"]}},
]
for task in TASKS:
    print(task["id"], "\nRequest:", task["prompt"], "\nExpected:", task["expected"], "\n")

<local-workspace>/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
<local-workspace>/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


extract_order 
Request: Extract the order ID and revised delivery time from: "Order Z17 was due at 14:00; it will now arrive at 14:30." Return only JSON with keys order_id and revised_time. 
Expected: {'order_id': 'Z17', 'revised_time': '14:30'} 

extract_missing 
Request: Extract the order ID and revised delivery time from: "Order K82 is delayed. A new arrival time has not been given." Use null for missing information. Return only JSON with keys order_id and revised_time. 
Expected: {'order_id': 'K82', 'revised_time': None} 

box_count 
Request: A van starts with 12 boxes. The driver unloads 5 boxes at a delivery, then picks up 3 additional boxes and loads them onto the van. How many boxes are on the van now? Return only JSON with key boxes and an integer value. 
Expected: {'boxes': 10} 

visit_order 
Request: Visit A, B, C, and D exactly once. A must be before B. C must be before A. D must be after B. Return only JSON with key route and a list of the four letters in a valid order. 
E

## 2. Models, environment, and execution settings

Use a GPU if available; CPU execution is supported but may be slow. Load one model at a time. Model weights stay in the Hugging Face cache, outside this folder. The two downloads total approximately 2.6 GB; memory use depends on precision and device. Initial loading and downloads are excluded from generation timings.

We pin each model and its tokenizer to a repository revision. We use the same sampling settings in all three conditions and reset the random seed before each generation. This makes the run easier to reproduce, but does not guarantee identical outputs across hardware or software. Qwen recommends sampling rather than greedy decoding for thinking mode. We use its thinking-mode settings for all conditions to keep sampling fixed; these differ from its separately recommended direct-mode settings. Our 2,048-token thinking cap is a resource limit, below the model card's general recommendation; failures at the cap are inconclusive about full-budget capability. This notebook uses one run per task/condition: timings and correctness are descriptive observations, not estimates of a general advantage.

Thinking gets a larger token limit so it has space for intermediate output. This deliberately differs from the direct-response limit; report the limits when interpreting cost or completeness.

In [2]:
REVISIONS = {'Qwen/Qwen3-0.6B': 'c1899de289a04d12100db370d81485cdf75e47ca', 'Qwen/Qwen3-0.6B-Base': 'da87bfb608c14b7cf20ba1ce41287e8de496c0cd'}
BASE = "Qwen/Qwen3-0.6B-Base"
POST = "Qwen/Qwen3-0.6B"
DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
DTYPE = torch.float16 if DEVICE != "cpu" else torch.float32
LIMITS = {"base":256, "instruct_direct":256, "instruct_thinking":2048}
RUN_INFO = {"started_utc":datetime.now(timezone.utc).isoformat(), "python":platform.python_version(), "torch":torch.__version__, "transformers":version("transformers"), "device":DEVICE, "dtype":str(DTYPE), "revisions":REVISIONS, "limits":LIMITS, "decoding":"sampling", "seed":42, "temperature":0.6, "top_p":0.95, "top_k":20, "repetitions":1}
print(json.dumps(RUN_INFO, indent=2))

def sync():
    if DEVICE == "mps": torch.mps.synchronize()
    elif DEVICE == "cuda": torch.cuda.synchronize()

def load(checkpoint):
    tokenizer = AutoTokenizer.from_pretrained(checkpoint, revision=REVISIONS[checkpoint])
    model = AutoModelForCausalLM.from_pretrained(checkpoint, revision=REVISIONS[checkpoint], dtype=DTYPE).to(DEVICE).eval()
    RUN_INFO.setdefault("generation_configs", {})[checkpoint] = model.generation_config.to_dict()
    return tokenizer, model

def release():
    gc.collect()
    if DEVICE == "mps": torch.mps.empty_cache()
    elif DEVICE == "cuda": torch.cuda.empty_cache()

results = []

{
  "started_utc": "2026-11-11T20:59:21.467017+00:00",
  "python": "3.9.6",
  "torch": "2.8.0",
  "transformers": "4.57.6",
  "device": "mps",
  "dtype": "torch.float16",
  "revisions": {
    "Qwen/Qwen3-0.6B": "c1899de289a04d12100db370d81485cdf75e47ca",
    "Qwen/Qwen3-0.6B-Base": "da87bfb608c14b7cf20ba1ce41287e8de496c0cd"
  },
  "limits": {
    "base": 256,
    "instruct_direct": 256,
    "instruct_thinking": 2048
  },
  "decoding": "sampling",
  "seed": 42,
  "temperature": 0.6,
  "top_p": 0.95,
  "top_k": 20,
  "repetitions": 1
}


## 3. Shared generation and checks

The runner saves the **exact rendered input**, raw generated text, decoded text, and generated token IDs. Input tokens are removed from the displayed continuation. Timings include the generation call and device synchronization, but exclude input preparation and decoding. They are neither time-to-first-token measurements nor hosted-service benchmarks.

For this model's exposed thinking format, `</think>` separates the reasoning portion from the final answer. We preserve the complete raw output. An absent closing marker in thinking mode is treated as an unfinished reasoning trace; we do not grade that trace as a final answer. Token counts include special tokens and the closing delimiter, where present.

The checker distinguishes valid JSON, exact requested JSON structure, and correct requested values. For the expected-value check on non-JSON responses, we inspect the **first decodable JSON object**. A matching object inside extra prose can contain the expected values while violating the format instruction. This recovery rule cannot resolve contradictory objects or claims; those require manual inspection. These checks do not judge faithfulness of reasoning or factual reliability beyond the predefined task answers.

In [3]:
def assess(final, expected):
    try:
        obj = json.loads(final.strip()); valid_json = True
    except (ValueError, TypeError):
        valid_json = False; obj = None
        # Recover an embedded object only for a separate semantic check.
        for match in re.finditer(r"\{", final):
            try:
                obj, _ = json.JSONDecoder().raw_decode(final[match.start():]); break
            except ValueError: pass
    correct_values = isinstance(obj, dict) and all(k in obj and type(obj[k]) is type(v) and obj[k] == v for k,v in expected.items())
    exact_structure = valid_json and isinstance(obj, dict) and set(obj) == set(expected)
    return {"valid_json":valid_json, "format_ok":exact_structure, "correct_values":bool(correct_values), "success":bool(exact_structure and correct_values)}

assert assess('{"boxes":10}', {"boxes":10})["success"]
assert not assess('{"boxes":true}', {"boxes":1})["success"]
assert assess('Answer: {"boxes":10}', {"boxes":10})["correct_values"]
assert not assess('Answer: {"boxes":10}', {"boxes":10})["format_ok"]

def generate(tokenizer, model, rendered, cap, thinking=False):
    inputs = tokenizer(rendered, add_special_tokens=False, return_tensors="pt").to(DEVICE)
    torch.manual_seed(42)
    sync(); start = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=cap, do_sample=True, temperature=0.6, top_p=0.95, top_k=20, pad_token_id=tokenizer.eos_token_id)
    sync(); elapsed = time.perf_counter() - start
    ids = output[0, inputs.input_ids.shape[1]:].tolist()
    raw = tokenizer.decode(ids, skip_special_tokens=False)
    clean = tokenizer.decode(ids, skip_special_tokens=True)
    marker = tokenizer.encode("</think>", add_special_tokens=False)
    ends = [i+len(marker) for i in range(len(ids)-len(marker)+1) if ids[i:i+len(marker)] == marker]
    split = ends[0] if thinking and ends else None
    final = tokenizer.decode(ids[split:], skip_special_tokens=True).strip() if split else ("" if thinking else clean.strip())
    eos = model.generation_config.eos_token_id
    eos = [eos] if isinstance(eos, int) else (eos or [])
    return {"rendered_input":rendered, "input_tokens":inputs.input_ids.shape[1], "raw_output":raw, "generated_ids":ids, "final_answer":final, "generated_tokens":len(ids), "reasoning_tokens_including_delimiter":split if thinking else 0, "final_tokens_including_stop":len(ids)-split if split else (None if thinking else len(ids)), "reasoning_closed":bool(split) if thinking else None, "seconds":round(elapsed,3), "ended_with_eos":bool(ids and ids[-1] in eos), "hit_token_limit":len(ids)>=cap, "max_new_tokens":cap}

def record(tokenizer, model, task, condition, rendered):
    row = {"task_id":task["id"], "kind":task["kind"], "condition":condition, "checkpoint":BASE if condition == "base" else POST, "expected":task["expected"]}
    row.update(generate(tokenizer, model, rendered, LIMITS[condition], thinking=condition=="instruct_thinking"))
    row.update(assess(row["final_answer"], task["expected"]))
    results.append(row)
    print(task["id"], condition, "seconds=",row["seconds"], "tokens=",row["generated_tokens"], "success=",row["success"], "limit=",row["hit_token_limit"])


In [4]:
from html import escape
from IPython.display import HTML

def show_task_results(condition):
    """Show saved or newly generated task responses and their separate checks."""
    labels = {"base": "Base checkpoint", "instruct_direct": "Post-trained: thinking disabled", "instruct_thinking": "Post-trained: thinking enabled"}
    for row in (r for r in results if r["condition"] == condition):
        task = next(t for t in TASKS if t["id"] == row["task_id"])
        esc = lambda value: escape(str(value))
        expected = json.dumps(task["expected"], indent=2, ensure_ascii=False)
        verdict = "PASS" if row["success"] else "FAIL"
        response = row["final_answer"] or "[No final answer was separated from the thinking trace.]"
        content = (f"<h3>{esc(task['id'])} — {esc(labels[condition])}</h3>"
                   f"<p><strong>Task:</strong> {esc(task['prompt'])}</p>"
                   f"<p><strong>Expected JSON:</strong></p><pre>{esc(expected)}</pre>"
                   f"<p><strong>Model response used by checker:</strong></p><pre style='white-space:pre-wrap'>{esc(response)}</pre>"
                   f"<p><strong>Verdict: {verdict}</strong> · First parsed object's values match: {row['correct_values']}"
                   f" · JSON-only structure: {row['format_ok']} · Token limit reached: {row['hit_token_limit']}</p>"
                   "<p>The value check can find an object within prose and does not resolve contradictions. "
                   "Passing requires both matching values and the exact JSON structure.</p>")
        if condition == "instruct_thinking":
            content += f"<details><summary>Full generated response, including thinking</summary><pre style='white-space:pre-wrap'>{esc(row['raw_output'])}</pre></details>"
        display(HTML(content))


## 4. Experiment A: the base checkpoint

We first send each request as plain text. A base model can recognize question/answer patterns, but there is no assumption that it will act as an assistant or obey our JSON instruction. A poor result here describes this request and input convention, not every possible way to prompt a base model.

We also try an unfinished passage and one request with demonstrations. The latter changes the input, so it is an illustration of in-context steering, not a controlled base-versus-instruct quality comparison.

In [5]:
tokenizer, model = load(BASE)
# A short warm-up is excluded from recorded task timings.
_ = generate(tokenizer, model, "The sky is", 8)
for task in TASKS:
    record(tokenizer, model, task, "base", task["prompt"])


extract_order base seconds= 2.036 tokens= 66 success= False limit= False
extract_missing base seconds= 1.391 tokens= 53 success= False limit= False
box_count base seconds= 7.902 tokens= 244 success= False limit= False
visit_order base seconds= 7.752 tokens= 256 success= False limit= True


In [6]:
show_task_results("base")

### Base model - `continuation` and `few-shot` prompting examples

These continuation and few-shot examples are separate from the four evaluated tasks. They illustrate how the base model responds to different prompts and are not included in the task scores.


In [7]:
base_examples = []
for label, prompt in [
    ("continuation", "The delivery driver checked the address and"),
    ("few_shot", 'Text: Order P10 will arrive at 09:15.\nJSON: {"order_id":"P10","revised_time":"09:15"}\nText: Order J20 is delayed; no new time is available.\nJSON: {"order_id":"J20","revised_time":null}\nText: Order Z17 was due at 14:00; it will now arrive at 14:30.\nJSON:')]:
    row = {"label":label}; row.update(generate(tokenizer, model, prompt, 128)); base_examples.append(row)
    print("<<<<<<<", label, ">>>>>>>", "\nINPUT:\n",prompt,"\nCONTINUATION:\n",row["raw_output"], "\n")

<<<<<<< continuation >>>>>>> 
INPUT:
 The delivery driver checked the address and 
CONTINUATION:
  found it was in the city of Napa. He decided to call the post office and ask for a map of the city. The map had a scale of 1 inch = 5 miles. The post office gave him a map with a scale of 1 inch = 10 miles. The post office told him that the city of Napa was 50 miles from the post office. How many inches long was the map he received?
To determine the length of the map received by the delivery driver, we need to understand the relationship between the scale of the maps and the actual distance between the city of Napa and the 

<<<<<<< few_shot >>>>>>> 
INPUT:
 Text: Order P10 will arrive at 09:15.
JSON: {"order_id":"P10","revised_time":"09:15"}
Text: Order J20 is delayed; no new time is available.
JSON: {"order_id":"J20","revised_time":null}
Text: Order Z17 was due at 14:00; it will now arrive at 14:30.
JSON: 
CONTINUATION:
  {"order_id":"Z17","revised_time":"14:30"}
Text: Order Q17 was due

In [8]:
del model, tokenizer
release()

## 5. Experiment B: one post-trained checkpoint, two modes

Load the post-trained model once. It has both instruction-following and extended-thinking capabilities. `enable_thinking` is an option supported by this model's chat template; it is not a universal Transformers switch and does not change the weights.

Inspect the exact template outputs before generating. Then run the same four user requests in direct and thinking modes. The base comparison now spans different training histories **and** appropriate input formats; it is a practical behavior comparison rather than an isolation of one training intervention.

In [9]:
tokenizer, model = load(POST)
def render(prompt, thinking):
    return tokenizer.apply_chat_template([{"role":"user", "content":prompt}], tokenize=False, add_generation_prompt=True, enable_thinking=thinking)
for thinking in [False, True]:
    print("THINKING", thinking, "\n", repr(render(TASKS[0]["prompt"], thinking)), "\n")
for condition, thinking in [("instruct_direct",False),("instruct_thinking",True)]:
    _ = generate(tokenizer, model, render("Say hello briefly.", thinking), 8, thinking=thinking)
    for task in TASKS:
        record(tokenizer, model, task, condition, render(task["prompt"], thinking))
del model, tokenizer
release()

THINKING False 
 '<|im_start|>user\nExtract the order ID and revised delivery time from: "Order Z17 was due at 14:00; it will now arrive at 14:30." Return only JSON with keys order_id and revised_time.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n' 

THINKING True 
 '<|im_start|>user\nExtract the order ID and revised delivery time from: "Order Z17 was due at 14:00; it will now arrive at 14:30." Return only JSON with keys order_id and revised_time.<|im_end|>\n<|im_start|>assistant\n' 

extract_order instruct_direct seconds= 0.84 tokens= 30 success= False limit= False
extract_missing instruct_direct seconds= 0.615 tokens= 22 success= False limit= False
box_count instruct_direct seconds= 0.435 tokens= 15 success= False limit= False
visit_order instruct_direct seconds= 0.636 tokens= 23 success= False limit= False
extract_order instruct_thinking seconds= 10.093 tokens= 314 success= True limit= False
extract_missing instruct_thinking seconds= 7.691 tokens= 267 success= True limit

In [10]:
show_task_results("instruct_direct")

In [11]:
show_task_results("instruct_thinking")

## 6. Results: read correctness, compliance, and cost separately

A successful row means the requested JSON structure and predefined values both passed. It does not establish general competence or reasoning faithfulness. A token-limit flag means the model used the entire allowed output budget; check the raw output before interpreting failure. Reasoning tokens that remain unclosed cannot be separated reliably and are reported as unknown.

Compare direct and thinking rows within the same task. Do not conclude that one mode is consistently faster or more accurate from four examples and one execution each. The different token limits and the local execution conditions also matter.

In [12]:
headers = ["Task", "Condition", "JSON format", "Correct values", "Success", "Tokens", "Seconds", "At limit"]
lines = ["| " + " | ".join(headers) + " |", "|" + " --- |" * len(headers)]
for r in results:
    lines.append("| " + " | ".join(str(x) for x in [r["task_id"],r["condition"],r["format_ok"],r["correct_values"],r["success"],r["generated_tokens"],r["seconds"],r["hit_token_limit"]]) + " |")
display(Markdown("\n".join(lines)))
assert len(results) == len(TASKS)*3
assert {(r["task_id"],r["condition"]) for r in results} == {(t["id"],m) for t in TASKS for m in LIMITS}
assert all(r["generated_tokens"] == len(r["generated_ids"]) for r in results)
assert all(r["generated_tokens"] <= r["max_new_tokens"] for r in results)
for condition in LIMITS:
    rows=[r for r in results if r["condition"]==condition]
    print(condition, "successful tasks:", sum(r["success"] for r in rows),"/",len(rows), "; token-limited:",sum(r["hit_token_limit"] for r in rows))
payload = {"run_info":RUN_INFO, "tasks":TASKS, "results":results, "base_examples":base_examples}
Path("results.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Saved results.json with all inputs, raw continuations, token IDs, and checks.")

| Task | Condition | JSON format | Correct values | Success | Tokens | Seconds | At limit |
| --- | --- | --- | --- | --- | --- | --- | --- |
| extract_order | base | False | True | False | 66 | 2.036 | False |
| extract_missing | base | False | True | False | 53 | 1.391 | False |
| box_count | base | False | False | False | 244 | 7.902 | False |
| visit_order | base | False | False | False | 256 | 7.752 | True |
| extract_order | instruct_direct | False | True | False | 30 | 0.84 | False |
| extract_missing | instruct_direct | False | False | False | 22 | 0.615 | False |
| box_count | instruct_direct | False | False | False | 15 | 0.435 | False |
| visit_order | instruct_direct | False | False | False | 23 | 0.636 | False |
| extract_order | instruct_thinking | True | True | True | 314 | 10.093 | False |
| extract_missing | instruct_thinking | True | True | True | 267 | 7.691 | False |
| box_count | instruct_thinking | False | True | False | 152 | 4.194 | False |
| visit_order | instruct_thinking | False | True | False | 788 | 35.972 | False |

base successful tasks: 0 / 4 ; token-limited: 1
instruct_direct successful tasks: 0 / 4 ; token-limited: 0
instruct_thinking successful tasks: 2 / 4 ; token-limited: 0
Saved results.json with all inputs, raw continuations, token IDs, and checks.


### Observations from the saved run

The recorded run used the Apple GPU (MPS), float16, and the pinned Qwen3 0.6B checkpoints. These are observations from four tasks and one recorded seed, not general accuracy estimates.

| Condition | First parsed object's values match | Strict JSON + expected values | Token-limit hits |
| --- | --- | --- | --- |
| Base | 2 / 4 | 0 / 4 | 1 / 4 |
| Post-trained, thinking disabled | 1 / 4 | 0 / 4 | 0 / 4 |
| Same post-trained checkpoint, thinking enabled | 4 / 4 | 2 / 4 | 0 / 4 |

- **Extraction:** the base model included the right values but surrounded them with prose. Its few-shot continuation began with the desired JSON object, then continued generating additional examples. This illustrates why an answer appearing somewhere in a continuation is different from following the requested response contract.
- **Contradictory answers:** the base box-count response first supplied `20`, then reasoned to `10` and supplied further objects. The automatic recovery rule selects the first object and flags its values as incorrect. A human can see the later correct calculation and the contradiction; the checker does not adjudicate that conflict.
- **Direct mode:** all four outputs were fenced code blocks, violating our strict JSON-only requirement. The missing-time task also dropped the known order ID; the box count and route were wrong. These are failures of this checkpoint/configuration on these tasks, not definitions of non-thinking models.
- **Thinking mode:** all four traces closed and all four final answers contained the expected values. The arithmetic and route answers still used fences, so only the two extraction responses passed strict formatting. A readable trace and a correct answer do not establish trace faithfulness.
- **Local cost:** direct task generations took about 0.5–0.9 seconds and used 15–30 tokens; thinking task generations took about 4.5–37.5 seconds and used 152–788 tokens. These are single-run local observations, with fixed mode order and unequal output caps. They do not establish a general speed or quality ratio.

The result supports using extraction and the mode toggle as teaching examples. It does not support saying that base models cannot answer instructions, that reasoning always improves accuracy, or that this four-task result predicts production behavior.

### Model selection pilot

A larger SmolLM3 3B pilot caused severe memory pressure on this 16 GB machine and was stopped. Its task-created model downloads were removed. The final experiment uses the smaller Qwen3 pair; no results from the larger pilot are mixed into the table. Before the recorded experiment, a separate Qwen pilot exposed confusion around the word “collects”; the final arithmetic request explicitly describes loading additional boxes onto the van. The final tasks were fixed before this recorded run, but they are exploratory teaching material, not an untouched evaluation set.


## 7. Interpretation and next experiments

Read the recorded outputs before choosing examples for the article. Ask:

1. Did the base checkpoint respond as requested, continue the text, or do something else? Did examples in the prompt change its behavior?
2. Did the post-trained model follow the JSON instruction? Separate that from answering correctly.
3. In which tasks did thinking change the answer, and how many additional tokens did it use? Were any answers unfinished?
4. Which observations are specific to this model, prompt, or execution budget?

Neither visible intermediate text nor a correct final answer proves how the internal computation reached that answer. Mode selection is not training. This notebook does not train a model, establish a causal effect of SFT, or rank model families.

Optional follow-ups: predefine a larger task set before evaluating; repeat sampled generations with recorded seeds; repeat warmed timing runs in alternating mode order; vary only the output budget to investigate truncation. Keep exploratory examples separate from a later untouched evaluation set.

### References

- [Qwen3 release and training recipe](https://qwenlm.github.io/blog/qwen3/)
- [Base model card](https://huggingface.co/Qwen/Qwen3-0.6B-Base)
- [Post-trained model and thinking controls](https://huggingface.co/Qwen/Qwen3-0.6B)
- [Hugging Face chat templates](https://huggingface.co/docs/transformers/chat_templating)
- [Anthropic: reasoning traces and faithfulness](https://www.anthropic.com/research/reasoning-models-dont-say-think)

Original notebook code; referenced software, datasets, and weights retain their respective licenses. Qwen3 weights are Apache-2.0. No model weights are distributed here.